# دمج بصري ونصي برأس عشوائي مستقل

شغّل خلية الكود الوحيدة بعد تفعيل GPU. لا تحتاج إلى Internet أو إعادة استخراج الصور، لكن أبقِ مجلد التجربة السابقة كاملًا، وCSV الأصلي والقائمة المدققة. أوقف أي تدريب آخر قبل التشغيل.

## ما الذي ندرّبه؟

نحافظ على النموذج البصري والنصي دون تغيير، ونستخدم ميزاتهما المحفوظة: 256 بصرية ثم 256 نصية. يبدأ رأس 512→256→128→2 بأوزان عشوائية، مع BatchNorm وReLU وDropout0.3. لا تُنسخ أوزان مصنّف النصي إلى هذا الرأس. وجود الفرعين في البنية لا يثبت أنهما مؤثران؛ نقيس ذلك بعد التدريب.

يتدرب الرأس على صفوف Train ذات الرابط المتاح. عندما يغيب الرابط، يستخدم النظام المدمج التنبؤ البصري كما في التجربة السابقة. تُعرض النتائج على جميع الصور، وعلى مجموعتي الرابط المتاح والمفقود منفصلتين. نتيجة النصي عند غياب الرابط تعتمد على المدخل الافتراضي المحفوظ، ويجب وصفها بذلك.

## إعدادات محددة قبل التشغيل

- حتى10حقب للرأس فقط، مع batch1024 وAdamW ومعدل تعلم0.001، وweight decay0.0001، وlabel smoothing0.1.
- توقف بعد3حقب دون تحسن Macro-F1 على كامل Validation.
- الاختيار من الحقب المدرّبة رقم1فأعلى؛ الحقبة العشوائية0 تُعرض للتشخيص فقط.
- يُعرض النصي مع الرجوع للبصري كخط أساس إضافي حتى إذا كان أفضل من الدمج المدرّب. اختيار أفضل رأس مدرّب لا يعني ادعاء تفوقه.

تختلف هذه الإعدادات عن التجربة السابقة ذات التهيئة النصية، ولذلك لا ننسب الفروق إلى التهيئة وحدها. هذه تجربة إضافية محددة، ولا يوجد ضمان لتحسنها. نحتفظ بنتائج التجارب غير الناجحة أيضًا.

## الملفات والاستكمال

المصدر للقراءة فقط:
`/kaggle/working/qr_three_systems_frozen_v1`

المخرجات الجديدة:
`/kaggle/working/qr_three_systems_random_fusion_v1`

يفحص الكود بصمات المصدر والميزات والعضوية والترتيب قبل التدريب. إذا كانت دفعة ميزات ناقصة أو تالفة، يتوقف ولا يعيد حسابها بصمت. قد يستغرق تحميل الميزات وفحصها بعض الوقت، لكنه لا يشغّل MobileNet أوBERT لاستخراجها مجددًا.

عند الانقطاع أعد تشغيل الخلية بالإعدادات نفسها؛ تستأنف آخر حقبة مكتملة سليمة. تحتاج عدةGBمن RAM للميزات. لا يتم تدريب أو تقييم صور Test.

## اختبار مساهمة كل فرع

بعد اختيار الأوزان، وعلى Validation ذات الرابط المتاح فقط، يقارن الدفتر: الميزات السليمة، وتصفير البصري، وتصفير النصي، وتبديل البصري بين الصور، وتبديل النصي بين الصور. التبديل ثابت البذرة ولا يستخدم التسميات. لا تدخل هذه النتائج في اختيار الحقبة.

هذه تدخلات تشخيصية قد تنشئ مدخلات خارج توزيع التدريب. انخفاض الأداء يشير إلى حساسية للفرع المعطل، ولا يثبت سببيًا فهم الصورة أو الرابط. وإذا لم يظهر أثر للفرع البصري فلا ندعي أن الدمج يستفيد منه.

## النتائج والحفظ

الجدول يعرض visual وtext وfeature_fusion، مع text_with_visual_fallback كخط أساس إضافي. تصدر أفضل حقبة مدرّبة كملف رأس صغير، وكملف كامل بأوزان الأنظمة الثلاثة تحت الأسماء visual.* وtext.* وfusion.*. يحتاج الاستخدام اللاحق إلى تعريف البنية والـtokenizer والتحويلات من هذا الدفتر والدفتر السابق؛ الأوزان وحدها ليست برنامج تشغيل.

تُنشأ حزمة ZIP تضم أفضل رأس والأوزان الكاملة والـtokenizer والتقارير، دون feature_cache الكبير. حالة المُحسّن محفوظة في epoch_*.pt داخل المجلد؛ نزّلها أيضًا إذا أردت الاستكمال لاحقًا.

أرسل هذه الملفات بعد النهاية:
`validation_selected_comparison.csv`
`validation_modality_ablations.csv`
`completion.json`

أوزان BERT القديمة لا تحمل بصمة تقسيم التدريب، وTest سبق فتحه لتجارب البصري. وثّق تاريخ التدريب والاختيار قبل تقرير اختبار مستقل للأنظمة الثلاثة.

فُحصت الصياغة والمقاييس التجريبية وتجميع الميزات ومطابقة البصمات محليًا. لم يُنفذ GPU أو تُقرأ ميزات Kaggle الفعلية في بيئة المساعد.


In [ ]:
print('NEW EXPERIMENT: random feature-fusion head; reuse verified cached features.',flush=True)
import os,json,time,uuid,random,hashlib,zipfile,shutil,gc,math
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
from tqdm import tqdm
from sklearn.metrics import accuracy_score,f1_score,precision_score,recall_score,roc_auc_score,average_precision_score,confusion_matrix

CACHE_RUN=Path('/kaggle/working/qr_three_systems_frozen_v1')
ROOT=Path('/kaggle/working/qr_visual_fresh')
SOURCE_CSV=Path('/kaggle/input/datasets/feras2004/spliteoriginal/splitsOriginal.csv')
OUT=Path('/kaggle/working/qr_three_systems_random_fusion_v1')
EXPECTED_SIGNATURE='d4509181f834a16144ad45e24b8ce686a4e608e07a9846a880757c705739d1ff'
VISUAL_SHA='11f170e26ba4e96a8ceaba4cbac66d26d5b8829aad9f79a1e898377eda92dfab'
TEXT_SHA='63d4fbc8afcd15ee39df27ad9fc8eec966d365b8c9ee02cef4f99c8bc79a98a4'
CFG=dict(seed=42,epochs=10,patience=3,batch_size=1024,eval_batch_size=4096,
         lr=1e-3,weight_decay=1e-4,label_smoothing=0.1,clip=1.0,threshold=0.5,
         initialization='random',input_size=512,hidden_sizes=[256,128],dropout=0.3,
         train_rows='available URLs only; both frozen features concatenated visual then text',
         missing_url_policy='visual fallback',selection='best trained epoch >=1 by full Validation Macro-F1')
assert torch.cuda.is_available(),'Enable GPU. No source models or images need to be recomputed.'
DEVICE=torch.device('cuda:0');torch.set_num_threads(1)
print('GPU:',torch.cuda.get_device_name(0),flush=True)
OUT.mkdir(parents=True,exist_ok=True)


def file_hash(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
    return h.hexdigest()

def normalize_json(value):return json.loads(json.dumps(value,ensure_ascii=False,allow_nan=False))

def atomic_json(value,path):
    path=Path(path);tmp=path.with_name(path.name+'.writing')
    with tmp.open('w',encoding='utf-8') as f:
        json.dump(value,f,ensure_ascii=False,indent=2,allow_nan=False);f.flush();os.fsync(f.fileno())
    os.replace(tmp,path)

def data_signature(frame):
    h=hashlib.sha256()
    for row in tqdm(frame[['path','label','split','url','ok','sha256']].itertuples(index=False,name=None),
                    total=len(frame),desc='Verify membership',mininterval=2):
        h.update((json.dumps(row,ensure_ascii=False,separators=(',',':'))+'\n').encode())
    return h.hexdigest()

print('[1/5] Verify feature-cache identity and frozen source checkpoints...',flush=True)
source_config_path=CACHE_RUN/'config.json'
assert source_config_path.is_file(),'Restore the previous full folder qr_three_systems_frozen_v1.'
source_identity=json.loads(source_config_path.read_text())
assert source_identity['data_signature']==EXPECTED_SIGNATURE
assert source_identity['visual_sha256']==VISUAL_SHA and source_identity['text_sha256']==TEXT_SHA
assert source_identity['cfg']['max_len']==128 and source_identity['cfg']['feature_version']==1
assert source_identity['cfg']['training']=='freeze both source models; train fusion head on available-URL Train rows'
# Official BERT id2label has keys0/1; JSON normalization preserves their sorted
# byte representation, so the original feature-cache identifier is unchanged.
CACHE_ID=hashlib.sha256(json.dumps(source_identity,sort_keys=True,separators=(',',':')).encode()).hexdigest()
source_paths={'visual':CACHE_RUN/'source_backup/visual.pt','text':CACHE_RUN/'source_backup/text.pt'}
for name,path in source_paths.items():
    assert path.is_file(),f'Missing frozen source: {path}'
    assert file_hash(path)==({'visual':VISUAL_SHA,'text':TEXT_SHA}[name])
tokenizer_source=CACHE_RUN/'tokenizer'
assert tokenizer_source.is_dir(),'Keep the saved tokenizer folder for reproducibility.'
for name,sha in source_identity.get('tokenizer_files_sha256',{}).items():
    assert file_hash(tokenizer_source/name)==sha,f'Tokenizer changed: {name}'

print('[2/5] Verify original Train/Validation metadata...',flush=True)
audit=json.loads((ROOT/'audit.json').read_text());assert audit['data_signature']==EXPECTED_SIGNATURE
assert audit.get('train_ready') is True
frame=None
for manifest in [ROOT/'verified_mobilenet_inputs/splits_verified.parquet',
                 ROOT/'verified_mobilenet_inputs/splits_verified.csv.gz',ROOT/'splits_clean.parquet']:
    if not manifest.is_file():continue
    try:
        candidate=pd.read_parquet(manifest) if manifest.suffix=='.parquet' else pd.read_csv(manifest,keep_default_na=False)
        assert len(candidate)==1005700 and candidate.path.is_unique
        assert candidate.label.isin([0,1]).all() and candidate.ok.isin([0,1]).all()
        assert data_signature(candidate)==EXPECTED_SIGNATURE
        frame=candidate;break
    except Exception as exc:print(manifest.name,type(exc).__name__,str(exc)[:160],flush=True)
assert frame is not None,'Restore a valid audited manifest; do not regenerate a split.'
assert SOURCE_CSV.is_file() and file_hash(SOURCE_CSV)==audit['source_splits_sha256']
columns=['url','ok','label','domain','fold','split']
raw=pd.read_csv(SOURCE_CSV,keep_default_na=False,usecols=['path']+columns)
assert raw.path.is_unique and len(raw)==audit['original_samples']
keys=frame.source_path if 'source_path' in frame else frame.path
assert keys.is_unique
aligned=raw.set_index('path').loc[keys]
for column in columns:
    if column in ['ok','label','fold']:
        assert np.array_equal(pd.to_numeric(frame[column]).to_numpy(),pd.to_numeric(aligned[column]).to_numpy()),column
    else:
        assert np.array_equal(frame[column].fillna('').astype(str).to_numpy(),aligned[column].fillna('').astype(str).to_numpy()),column
assert frame.groupby('sha256').split.nunique().max()==1
assert frame.groupby('sha256').label.nunique().max()==1
assert frame.groupby('domain',dropna=False).split.nunique().max()==1
assert frame[frame.ok.eq(1)].groupby('url').split.nunique().max()==1
train=frame[frame.split.eq('train')].reset_index(drop=True)
val=frame[frame.split.eq('val')].reset_index(drop=True)
assert len(train)==710746 and len(val)==148529
for df in [train,val]:
    df['url']=df.url.fillna('').astype(str)
    df['available']=df.ok.eq(1)&df.url.str.strip().ne('')
del raw,aligned,frame,candidate;gc.collect()

print('[3/5] Read and verify cached features; no image extraction...',flush=True)

def load_cache(df,split):
    n=len(df);data=dict(features=np.empty((n,512),np.float32),visual_logits=np.empty((n,2),np.float32),
        text_logits=np.empty((n,2),np.float32),labels=df.label.to_numpy(dtype=np.int64),available=df.available.to_numpy(dtype=bool))
    shard_rows=int(source_identity['cfg']['shard_rows'])
    for start in tqdm(range(0,n,shard_rows),desc=split+' cache verification',mininterval=2):
        end=min(n,start+shard_rows)
        path=CACHE_RUN/'feature_cache'/f'{split}_{start:07d}_{end:07d}.npz'
        assert path.is_file() and path.with_suffix('.json').is_file(),f'Incomplete cached features: {path.name}. Finish the old extraction first.'
        receipt=json.loads(path.with_suffix('.json').read_text())
        assert receipt['cache_id']==CACHE_ID,'Cache identity differs. Do not bypass.'
        assert receipt['start']==start and receipt['end']==end
        assert file_hash(path)==receipt['sha256'],f'Corrupted shard: {path.name}'
        with np.load(path,allow_pickle=False) as shard:
            assert set(shard.files)=={'features','visual_logits','text_logits','labels','available'}
            assert shard['features'].shape==(end-start,512)
            assert shard['visual_logits'].shape==shard['text_logits'].shape==(end-start,2)
            assert np.array_equal(shard['labels'],data['labels'][start:end])
            assert np.array_equal(shard['available'],data['available'][start:end])
            for name in ['features','visual_logits','text_logits']:
                assert shard[name].dtype==np.float32 and np.isfinite(shard[name]).all()
                data[name][start:end]=shard[name]
    return data

val_data=load_cache(val,'val');train_data=load_cache(train,'train')
assert train_data['available'].sum()>1
identity=normalize_json(dict(cfg=CFG,source_cache_id=CACHE_ID,source_config_sha256=file_hash(source_config_path),
    data_signature=EXPECTED_SIGNATURE,visual_sha256=VISUAL_SHA,text_sha256=TEXT_SHA,
    torch_version=str(torch.__version__)))
config_path=OUT/'config.json'
if config_path.exists():assert json.loads(config_path.read_text())==identity,'New experiment settings changed; use a new OUT folder.'
else:atomic_json(identity,config_path)

random.seed(CFG['seed']);np.random.seed(CFG['seed']);torch.manual_seed(CFG['seed']);torch.cuda.manual_seed_all(CFG['seed'])
torch.backends.cudnn.benchmark=False;torch.backends.cudnn.deterministic=True

class FusionHead(nn.Sequential):
    def __init__(self):
        super().__init__(nn.Linear(512,256),nn.BatchNorm1d(256),nn.ReLU(),nn.Dropout(CFG['dropout']),
                         nn.Linear(256,128),nn.BatchNorm1d(128),nn.ReLU(),nn.Linear(128,2))

head=FusionHead().to(DEVICE)
# All input columns start randomly; no text-classifier weights are copied.
assert torch.count_nonzero(head[0].weight[:,:256])>0
assert torch.count_nonzero(head[0].weight[:,256:])>0
optimizer=torch.optim.AdamW(head.parameters(),lr=CFG['lr'],weight_decay=CFG['weight_decay'])
scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=CFG['epochs'],eta_min=1e-5)
criterion=nn.CrossEntropyLoss(label_smoothing=CFG['label_smoothing'])


def logits_probability(logits):
    shifted=logits-logits.max(axis=1,keepdims=True);exp=np.exp(shifted)
    return exp[:,1]/exp.sum(axis=1)


def metrics(y,p):
    if len(y)==0:return dict(samples=0)
    assert np.isfinite(p).all();pred=(p>=CFG['threshold']).astype(np.int64)
    return dict(samples=int(len(y)),accuracy=float(accuracy_score(y,pred)),
        macro_f1=float(f1_score(y,pred,average='macro',zero_division=0)),
        malicious_precision=float(precision_score(y,pred,zero_division=0)),
        malicious_recall=float(recall_score(y,pred,zero_division=0)),
        malicious_f1=float(f1_score(y,pred,zero_division=0)),
        roc_auc=float(roc_auc_score(y,p)) if len(np.unique(y))==2 else None,
        average_precision=float(average_precision_score(y,p)) if len(np.unique(y))==2 else None,
        confusion_matrix=confusion_matrix(y,pred,labels=[0,1]).tolist())


def infer_head(features):
    head.eval();parts=[]
    with torch.no_grad():
        for start in range(0,len(features),CFG['eval_batch_size']):
            x=torch.from_numpy(features[start:start+CFG['eval_batch_size']]).to(DEVICE)
            p=head(x).softmax(1)[:,1];assert torch.isfinite(p).all();parts.append(p.cpu().numpy())
    return np.concatenate(parts)

visual_p=logits_probability(val_data['visual_logits'])
text_p=logits_probability(val_data['text_logits'])
available=val_data['available'];labels=val_data['labels']
router_p=np.where(available,text_p,visual_p)

def evaluation(tag,save_predictions=False):
    head_p=infer_head(val_data['features'])
    fused_p=np.where(available,head_p,visual_p)
    systems={'visual':visual_p,'text':text_p,'text_with_visual_fallback':router_p,'feature_fusion':fused_p}
    rows=[]
    for name,p in systems.items():
        for group,mask in [('all',np.ones(len(labels),dtype=bool)),('url_available',available),('url_missing',~available)]:
            rows.append(dict(system=name,group=group,**metrics(labels[mask],p[mask])))
    pd.DataFrame(rows).drop(columns='confusion_matrix',errors='ignore').to_csv(OUT/f'validation_{tag}_comparison.csv',index=False)
    atomic_json(rows,OUT/f'validation_{tag}_metrics.json')
    for row in rows:
        if row['group']=='all':print(f"{tag} {row['system']}: Acc={row['accuracy']:.4%} | Macro-F1={row['macro_f1']:.4%} | AUC={row['roc_auc']:.5f}",flush=True)
    if save_predictions:
        predictions=val[['path','label','ok','available']].copy()
        for name,p in systems.items():predictions[name+'_probability']=p
        predictions.to_csv(OUT/f'validation_{tag}_predictions.csv.gz',index=False,compression='gzip')
    return next(row for row in rows if row['system']=='feature_fusion' and row['group']=='all')


def head_cpu():return {k:v.detach().cpu().clone() for k,v in head.state_dict().items()}


def verified_save(payload,label):
    path=OUT/(label+'_'+uuid.uuid4().hex[:12]+'.pt');tmp=path.with_suffix('.writing')
    with tmp.open('xb') as f:torch.save(payload,f);f.flush();os.fsync(f.fileno())
    with zipfile.ZipFile(tmp) as archive:assert archive.testzip() is None
    restored=torch.load(tmp,map_location='cpu',weights_only=False)
    for field in ['fusion_head_state','model_state']:
        if field in payload:
            assert set(restored[field])==set(payload[field])
            for key,value in payload[field].items():assert torch.equal(restored[field][key],value.cpu()),key
    if 'optimizer' in payload:
        assert restored['optimizer']['param_groups']==payload['optimizer']['param_groups']
        assert set(restored['optimizer']['state'])==set(payload['optimizer']['state'])
        for index,entry in payload['optimizer']['state'].items():
            for key,value in entry.items():
                saved=restored['optimizer']['state'][index][key]
                if torch.is_tensor(value):assert torch.equal(saved,value.cpu())
                else:assert saved==value
    if 'fusion_head_state' in restored:
        state_rng=torch.get_rng_state();probe=FusionHead().eval()
        probe.load_state_dict(restored['fusion_head_state'],strict=True);torch.set_rng_state(state_rng)
        del probe
    sha=file_hash(tmp);os.replace(tmp,path)
    atomic_json(dict(file=path.name,sha256=sha,epoch=payload['epoch']),path.with_suffix('.json'))
    print('PASS: verified',path.name,flush=True)
    return path


def payload(epoch,best,best_file,stale,history,validation=None,full=True):
    cp=dict(fusion_head_state=head_cpu(),epoch=epoch,identity=identity,validation=validation)
    if full:cp.update(optimizer=optimizer.state_dict(),scheduler=scheduler.state_dict(),best_macro_f1=best,
        best_file=best_file,stale=stale,history=history,python_rng=random.getstate(),numpy_rng=np.random.get_state(),
        torch_rng=torch.get_rng_state(),cuda_rng=torch.cuda.get_rng_state_all())
    return cp

start_epoch=0;best=-1.0;best_file=None;stale=0;history=[]
receipts=[]
for path in OUT.glob('epoch_*.json'):
    record=json.loads(path.read_text());receipts.append((int(record['epoch']),record))
if receipts:
    resumed=False
    for _,record in sorted(receipts,key=lambda x:x[0],reverse=True):
        try:
            path=OUT/record['file'];assert file_hash(path)==record['sha256']
            cp=torch.load(path,map_location='cpu',weights_only=False);assert cp['identity']==identity and cp['epoch']>=1
            best_path=OUT/cp['best_file'];best_receipt=json.loads(best_path.with_suffix('.json').read_text())
            assert file_hash(best_path)==best_receipt['sha256']
        except Exception as exc:print('Cannot resume',record['file'],type(exc).__name__,flush=True);continue
        head.load_state_dict(cp['fusion_head_state'],strict=True);optimizer.load_state_dict(cp['optimizer']);scheduler.load_state_dict(cp['scheduler'])
        start_epoch=cp['epoch'];best=cp['best_macro_f1'];best_file=cp['best_file'];stale=cp['stale'];history=cp['history']
        random.setstate(cp['python_rng']);np.random.set_state(cp['numpy_rng']);torch.set_rng_state(cp['torch_rng']);torch.cuda.set_rng_state_all(cp['cuda_rng'])
        del cp;resumed=True;print('Resumed trained fusion after epoch',start_epoch,flush=True);break
    assert resumed,'No valid full checkpoint; inspect files. Old experiment/cache are unchanged.'
else:
    print('Random epoch0 is diagnostic only and cannot be selected.',flush=True)
    evaluation('random_initial',save_predictions=True)
    # Test one actual optimizer step and restore before real training.
    original_head=head_cpu();state_py=random.getstate();state_np=np.random.get_state();state_t=torch.get_rng_state();state_cuda=torch.cuda.get_rng_state_all()
    idx=np.flatnonzero(train_data['available'])[:32];assert len(idx)>1
    head.train();x=torch.from_numpy(train_data['features'][idx]).to(DEVICE);y=torch.from_numpy(train_data['labels'][idx]).to(DEVICE)
    optimizer.zero_grad(set_to_none=True);loss=criterion(head(x),y);assert torch.isfinite(loss);loss.backward()
    nn.utils.clip_grad_norm_(head.parameters(),CFG['clip'],error_if_nonfinite=True);optimizer.step()
    verified_save(payload(-1,-1,None,0,[]),'storage_smoke')
    head.load_state_dict(original_head,strict=True);optimizer.state.clear();optimizer.zero_grad(set_to_none=True)
    random.setstate(state_py);np.random.set_state(state_np);torch.set_rng_state(state_t);torch.cuda.set_rng_state_all(state_cuda)
    del original_head,x,y,loss

print('[4/5] Train RANDOM fusion head; source backbones remain frozen...',flush=True)
train_indices=np.flatnonzero(train_data['available'])
for epoch in range(start_epoch+1,CFG['epochs']+1):
    if stale>=CFG['patience']:break
    head.train();order=np.random.permutation(train_indices);seen=0;correct=0;total_loss=0.;started=time.time()
    batches=[order[i:i+CFG['batch_size']] for i in range(0,len(order),CFG['batch_size'])]
    if len(batches)>1 and len(batches[-1])==1:
        final=batches.pop();batches[-1]=np.concatenate([batches[-1],final])
    for step,idx in enumerate(tqdm(batches,desc=f'Fusion {epoch}/{CFG["epochs"]}',mininterval=1),1):
        x=torch.from_numpy(train_data['features'][idx]).to(DEVICE);y=torch.from_numpy(train_data['labels'][idx]).to(DEVICE)
        optimizer.zero_grad(set_to_none=True);logits=head(x);loss=criterion(logits,y)
        assert torch.isfinite(loss);loss.backward()
        nn.utils.clip_grad_norm_(head.parameters(),CFG['clip'],error_if_nonfinite=True);optimizer.step()
        seen+=len(idx);correct+=int((logits.detach().argmax(1)==y).sum());total_loss+=float(loss.detach())*len(idx)
        if step==1 or step%50==0 or step==len(batches):
            speed=seen/max(time.time()-started,1e-6)
            print(f'Epoch {epoch}: {seen:,}/{len(order):,} ({seen/len(order):.1%}) | loss={total_loss/seen:.4f} | train_acc={correct/seen:.2%} | {speed:.0f} rows/s',flush=True)
    assert seen==len(train_indices)
    score=evaluation(f'epoch_{epoch:02d}');improved=score['macro_f1']>best+1e-6
    history.append(dict(epoch=epoch,train_loss=total_loss/seen,train_accuracy=correct/seen,validation=score,improved=improved))
    if improved:
        best=score['macro_f1'];stale=0
        best_file=verified_save(payload(epoch,best,None,0,history,score,full=False),f'best_epoch_{epoch:02d}').name
    else:stale+=1
    scheduler.step()
    verified_save(payload(epoch,best,best_file,stale,history,score),f'epoch_{epoch:02d}')
    atomic_json(history,OUT/'history.json')
    print(f'Best TRAINED fusion Macro-F1={best:.4%}; no improvement {stale}/{CFG["patience"]}',flush=True)

print('[5/5] Restore selected trained head, ablations and full-model export...',flush=True)
assert best_file is not None
selected_path=OUT/best_file;receipt=json.loads(selected_path.with_suffix('.json').read_text())
assert file_hash(selected_path)==receipt['sha256']
selected=torch.load(selected_path,map_location='cpu',weights_only=False);assert selected['epoch']>=1
head.load_state_dict(selected['fusion_head_state'],strict=True)
selected_score=evaluation('selected',save_predictions=True)
assert abs(selected_score['macro_f1']-best)<1e-6

# Diagnostic only: same available-URL subset, no fallback masks the effect.
# Permutations use no labels. These interventions can be out-of-distribution.
z=val_data['features'][available];y=labels[available]
rng=np.random.default_rng(2026)
permutation_visual=rng.permutation(len(z));permutation_text=rng.permutation(len(z))
ablations=[]
for name in ['intact','zero_visual','zero_text','permute_visual','permute_text']:
    changed=z.copy()
    if name=='zero_visual':changed[:,:256]=0
    elif name=='zero_text':changed[:,256:]=0
    elif name=='permute_visual':changed[:,:256]=z[permutation_visual,:256]
    elif name=='permute_text':changed[:,256:]=z[permutation_text,256:]
    p=infer_head(changed);m=metrics(y,p);ablations.append(dict(intervention=name,**m))
    print(f"Ablation {name}: Acc={m['accuracy']:.4%} | Macro-F1={m['macro_f1']:.4%}",flush=True)
    del changed
pd.DataFrame(ablations).drop(columns='confusion_matrix').to_csv(OUT/'validation_modality_ablations.csv',index=False)
atomic_json(ablations,OUT/'validation_modality_ablations.json')

# Export complete three-system weights without downloading/retraining BERT.
full_state={}
for name,path in source_paths.items():
    assert file_hash(path)==({'visual':VISUAL_SHA,'text':TEXT_SHA}[name])
    source_cp=torch.load(path,map_location='cpu',weights_only=False)
    for key,value in source_cp['model_state'].items():full_state[name+'.'+key]=value.detach().cpu().clone()
    del source_cp
for key,value in head_cpu().items():full_state['fusion.'+key]=value
export=verified_save(dict(model_state=full_state,fusion_head_state=head_cpu(),epoch=selected['epoch'],
    cfg=CFG,identity=identity,source_model_identity=source_identity,validation=selected_score,
    architecture='ThreeSystems: visual=MobileNetVisual, text=TextModel, fusion=512->256->128->2',
    missing_url_policy='visual fallback; feature head used only when URL is available'),
    f'selected_three_systems_epoch_{selected["epoch"]:02d}')

tokenizer_out=OUT/'tokenizer';tokenizer_out.mkdir(exist_ok=True)
for path in tokenizer_source.iterdir():
    if path.is_file():
        destination=tokenizer_out/path.name
        if destination.exists():assert file_hash(destination)==file_hash(path),'Existing tokenizer differs.'
        else:
            temp=destination.with_name(destination.name+'.writing');shutil.copy2(path,temp)
            assert file_hash(temp)==file_hash(path);os.replace(temp,destination)
router_score=metrics(labels,router_p)
atomic_json(dict(selected_trained_epoch=selected['epoch'],selected_head=str(selected_path),full_model=str(export),
    feature_fusion_validation=selected_score,router_baseline_validation=router_score,
    beats_router_macro_f1=bool(selected_score['macro_f1']>router_score['macro_f1']),
    ablations_group='Validation URL available only; descriptive diagnostics, not causal proof',
    test_evaluated=False,backbones_retrained=False,features_recomputed=False,
    legacy_text_provenance='Original text checkpoint has no data_signature; document old Train membership before independent-Test claims.'),OUT/'completion.json')
# Bundle review/deployment files only, excluding the large feature cache.
zip_path=OUT/f'fusion_random_export_{time.strftime("%Y%m%d_%H%M%S")}_{uuid.uuid4().hex[:6]}.zip'
files=[export,export.with_suffix('.json'),selected_path,selected_path.with_suffix('.json'),
       OUT/'config.json',OUT/'history.json',OUT/'completion.json',OUT/'validation_selected_comparison.csv',
       OUT/'validation_selected_metrics.json',OUT/'validation_selected_predictions.csv.gz',
       OUT/'validation_modality_ablations.csv',OUT/'validation_modality_ablations.json']
files.extend(path for path in tokenizer_out.iterdir() if path.is_file())
backup_manifest=[dict(file=str(p.relative_to(OUT)),sha256=file_hash(p),bytes=p.stat().st_size) for p in files]
tmp=zip_path.with_suffix('.writing')
with zipfile.ZipFile(tmp,'x',compression=zipfile.ZIP_DEFLATED,compresslevel=1) as archive:
    for path in files:archive.write(path,arcname=str(path.relative_to(OUT)))
    archive.writestr('backup_manifest.json',json.dumps(backup_manifest,indent=2))
with zipfile.ZipFile(tmp) as archive:
    assert archive.testzip() is None
    for entry in backup_manifest:
        h=hashlib.sha256()
        with archive.open(entry['file']) as f:
            for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
        assert h.hexdigest()==entry['sha256']
os.replace(tmp,zip_path)
print('Selected trained head:',selected_path,flush=True)
print('Full three-system weights:',export,flush=True)
print('Results:',OUT,flush=True)
print('Verified ZIP:',zip_path,flush=True)
print('No source weights or cached features were changed. No Test evaluation.',flush=True)
